In [1]:
# mypy: ignore-errors
# ruff: noqa
import gc
import math
import os
import sys
import warnings

import numpy as np
import pandas as pd
import torch
from tqdm import tqdm

warnings.simplefilter("ignore")
from sklearn.model_selection import KFold

current_dir = os.getcwd()
parent_dir = os.path.abspath(os.path.join(current_dir, ".."))
sys.path.append(parent_dir)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [10]:
# ruff: noqa
%load_ext autoreload
%autoreload 2
from drGAT import drGAT
from drGAT.myutils import *
from drGAT.load_data import load_data
from drGAT.sampler import NewSampler
from get_params import get_params
from metrics import compute_metrics_stats

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [3]:
name = 'nci'

In [4]:
(
    res,
    pos_num,
    null_mask,
    drug_sim,
    cell_sim,
    gene_sim,
    A_cg,
    A_dg,
    _,
    _,
    _,
) = load_data(name)
res = res.T
cell_sum = np.sum(res, axis=1)
drug_sum = np.sum(res, axis=0)

target_dim = [
    0,  # Cell
    # 1  # Drug
]

load nci
unique drugs: 177
unique genes: 251
DTI unique genes:  251
Top 90% variable genes:  2383
Total:  2582
Final gene exp shape: (60, 2582)
Final drug Act shape: (1005, 60)


100%|██████████| 25/25 [00:02<00:00,  8.76it/s]

Done!


In [5]:
n_kfold = 1
true_datas = pd.DataFrame()
predict_datas = pd.DataFrame()
for dim in target_dim:
    for seed, target_index in tqdm(enumerate(np.arange(res.shape[dim]))):
        if dim:
            if drug_sum[target_index] < 10:
                continue
        else:
            if cell_sum[target_index] < 10:
                continue
        epochs = []
        true_data_s = pd.DataFrame()
        predict_data_s = pd.DataFrame()
        sampler = NewSampler(
            res,
            null_mask.T.values,
            target_dim=dim,
            target_index=target_index,
            S_d=drug_sim,
            S_c=cell_sim,
            S_g=gene_sim,
            A_cg=A_cg,
            A_dg=A_dg,
            PATH='../nci_data/',
            seed=seed,
        )
        break

0it [00:00, ?it/s]


In [6]:
import argparse

from tqdm import tqdm

In [21]:

def load_data(args):
    """Load data based on the specified dataset."""
    if args.data == "gdsc1":
        print("load gdsc1")
        PATH = "gdsc1_data/"
        return _load_data(PATH)
    elif args.data == "gdsc2":
        print("load gdsc2")
        PATH = "gdsc2_data/"
        return _load_data(PATH)
    elif args.data == "ctrp":
        PATH = "ctrp_data/"
        return _load_data(PATH, is_ctrp=True)
    elif args.data == "nci":
        print("load nci")
        PATH = "nci_data/"
        return _load_nci(PATH)
    else:
        raise NotImplementedError


def _get_base_data(PATH):
    """Load and prepare base data common to all datasets."""
    # Load original drug response data
    drugAct = pd.read_csv(PATH + "drugAct.csv", index_col=0)

    # Load and concatenate gene expression data
    exprs = pd.concat(
        [
            pd.read_csv(PATH + "gene_exp_part1.csv.gz", index_col=0),
            pd.read_csv(PATH + "gene_exp_part2.csv.gz", index_col=0),
        ]
    ).T.dropna()

    return drugAct, exprs


def _load_data(PATH, is_ctrp=False):
    data_dir = dir_path(k=1) + PATH
    # 加载细胞系-药物矩阵

    drugAct, exprs = _get_base_data(data_dir)
    cells = sorted(
        set(drugAct.columns)
        & set(exprs.index)
        & set(pd.read_csv(data_dir + "mut.csv", index_col=0).T.index)
    )

    SMILES = pd.read_csv(data_dir + "drug2smiles.csv", index_col=0)
    exprs = exprs.loc[cells]
    drugAct = drugAct.loc[sorted(SMILES.drugs), cells]
    exprs = np.array(exprs, dtype=np.float32)

    if is_ctrp:
        drugAct = drugAct.apply(lambda x: (x - np.nanmean(x)) / np.nanstd(x))

    # Convert drug activity to binary response matrix
    res = (drugAct > 0).astype(int)
    res = np.array(res, dtype=np.float32).T

    pos_num = sp.coo_matrix(res).data.shape[0]

    # 加载药物-指纹特征矩阵
    drug_feature = pd.read_csv(
        data_dir + "nih_drug_feature.csv", index_col=0, header=0
    ).loc[sorted(SMILES.drugs)]
    drug_feature = np.array(drug_feature, dtype=np.float32)

    null_mask = (drugAct.isna()).astype(int).T
    null_mask = np.array(null_mask, dtype=np.float32)
    return res, drug_feature, exprs, null_mask, pos_num


def _load_nci(PATH):
    data_dir = dir_path(k=1) + PATH
    # 加载细胞系-药物矩阵

    drugAct, exprs = _get_base_data(data_dir)
    drugAct.columns = exprs.index
    cells = sorted(
        set(drugAct.columns)
        & set(exprs.index)
        & set(pd.read_csv(data_dir + "mut.csv", index_col=0).T.index)
    )

    # Load mechanism of action (moa) data
    moa = pd.read_csv("../Figs/nsc_cid_smiles_class_name.csv", index_col=0)

    # Filter drugs that have SMILES information
    drugAct = drugAct[drugAct.index.isin(moa.NSC)]

    # Load drug synonyms and filter based on availability in other datasets
    tmp = pd.read_csv("../data/drugSynonym.csv")
    tmp = tmp[
        (~tmp.nci60.isna() & ~tmp.ctrp.isna())
        | (~tmp.nci60.isna() & ~tmp.gdsc1.isna())
        | (~tmp.nci60.isna() & ~tmp.gdsc2.isna())
    ]
    tmp = [int(i) for i in set(tmp["nci60"].str.split("|").explode())]

    # Select drugs not classified as 'Other' in MOA and included in other datasets
    drugAct = drugAct.loc[
        sorted(
            set(drugAct.index)
            & (set(moa[moa["MECHANISM"] != "Other"]["NSC"]) | set(tmp))
        )
    ]

    # SMILES = pd.read_csv(data_dir + "drug2smiles.csv", index_col=0)
    exprs = exprs.loc[cells]
    drugAct = drugAct.loc[:, cells]
    exprs = np.array(exprs, dtype=np.float32)

    # Convert drug activity to binary response matrix
    res = (drugAct > 0).astype(int)
    res = np.array(res, dtype=np.float32).T

    pos_num = sp.coo_matrix(res).data.shape[0]

    # 加载药物-指纹特征矩阵
    # drug_feature = pd.read_csv(data_dir + "nih_drug_feature.csv", index_col=0, header=0)
    # drug_feature = np.array(drug_feature, dtype=np.float32)

    null_mask = (drugAct.isna()).astype(int).T
    null_mask = np.array(null_mask, dtype=np.float32)
    return res, exprs, null_mask, pos_num


In [22]:
class Args:
    def __init__(self):
        self.device = torch.device(
            "cuda" if torch.cuda.is_available() else "cpu"
        )  # cuda:number or cpu
        self.data = "nci"  # Dataset{gdsc or ccle}
        self.lr = 0.001  # the learning rate
        self.wd = 1e-5  # the weight decay for l2 normalizaton
        self.layer_size = [1024, 1024]  # Output sizes of every layer
        self.alpha = 0.25  # the scale for balance gcn and ni
        self.gamma = 8  # the scale for sigmod
        self.epochs = 10  # the epochs for model


args = Args()

In [23]:
res, exprs, null_mask, pos_num = load_data(args)
cell_sum = np.sum(res, axis=1)
drug_sum = np.sum(res, axis=0)

target_dim = [
    0,  # Cell
    # 1  # Drug
]

load nci


In [24]:
class OriginalNewSampler(object):
    def __init__(self, original_adj_mat, null_mask, target_dim, target_index, seed):
        super().__init__()
        self.seed = seed
        self.set_seed()
        self.adj_mat = original_adj_mat
        self.null_mask = null_mask
        self.dim = target_dim
        self.target_index = target_index
        self.train_data, self.test_data = self.sample_train_test_data()
        self.train_mask, self.test_mask = self.sample_train_test_mask()

    def set_seed(self):
        np.random.seed(self.seed)  # NumPyのシードを設定
        torch.manual_seed(self.seed)  # PyTorchのシードを設定

    def sample_target_test_index(self):
        if self.dim:
            target_pos_index = np.where(self.adj_mat[:, self.target_index] == 1)[0]
        else:
            target_pos_index = np.where(self.adj_mat[self.target_index, :] == 1)[0]
        return target_pos_index

    def sample_train_test_data(self):
        test_data = np.zeros(self.adj_mat.shape, dtype=np.float32)
        test_index = self.sample_target_test_index()
        if self.dim:
            test_data[test_index, self.target_index] = 1
        else:
            test_data[self.target_index, test_index] = 1
        train_data = self.adj_mat - test_data
        train_data = torch.from_numpy(train_data)
        test_data = torch.from_numpy(test_data)
        return train_data, test_data

    def sample_train_test_mask(self):
        test_index = self.sample_target_test_index()
        neg_value = np.ones(self.adj_mat.shape, dtype=np.float32)
        neg_value = neg_value - self.adj_mat - self.null_mask
        neg_test_mask = np.zeros(self.adj_mat.shape, dtype=np.float32)
        if self.dim:
            target_neg_index = np.where(neg_value[:, self.target_index] == 1)[0]
            if test_index.shape[0] < target_neg_index.shape[0]:
                target_neg_test_index = np.random.choice(
                    target_neg_index, test_index.shape[0], replace=False
                )
            else:
                target_neg_test_index = target_neg_index
            neg_test_mask[target_neg_test_index, self.target_index] = 1
            neg_value[:, self.target_index] = 0
        else:
            target_neg_index = np.where(neg_value[self.target_index, :] == 1)[0]
            if test_index.shape[0] < target_neg_index.shape[0]:
                target_neg_test_index = np.random.choice(
                    target_neg_index, test_index.shape[0], replace=False
                )
            else:
                target_neg_test_index = target_neg_index
            neg_test_mask[self.target_index, target_neg_test_index] = 1
            neg_value[self.target_index, :] = 0
        train_mask = (self.train_data.numpy() + neg_value).astype(bool)
        test_mask = (self.test_data.numpy() + neg_test_mask).astype(bool)
        train_mask = torch.from_numpy(train_mask)
        test_mask = torch.from_numpy(test_mask)
        return train_mask, test_mask

In [25]:
n_kfold = 1
true_data_s = pd.DataFrame()
predict_data_s = pd.DataFrame()
for dim in target_dim:
    for seed, target_index in enumerate(tqdm(np.arange(res.shape[dim]))):
        if dim:
            if drug_sum[target_index] < 10:
                continue
        else:
            if cell_sum[target_index] < 10:
                continue
        epochs = []
        originalsampler = OriginalNewSampler(res, null_mask, dim, target_index, seed)
        break

  0%|          | 0/60 [00:00<?, ?it/s]


In [30]:
np.alltrue((sampler.train_data == originalsampler.train_data).numpy())

True

tensor([[0., 0., 0.,  ..., 0., 0., 0.],
        [0., 0., 0.,  ..., 1., 1., 1.],
        [1., 0., 0.,  ..., 0., 0., 0.],
        ...,
        [0., 0., 0.,  ..., 0., 0., 0.],
        [1., 1., 1.,  ..., 1., 0., 1.],
        [1., 0., 0.,  ..., 1., 1., 1.]])